# SolIA : Évaluation de la solvabilité des vendeurs e-commerce

## 1. Introduction

Les vendeurs e-commerce présentent des revenus fluctuants et une forte saisonnalité,
ce qui complique leur évaluation par les méthodes bancaires traditionnelles.

### Problématique
Comment évaluer, à partir de données transactionnelles, la capacité d’un vendeur
e-commerce à rembourser un prêt bancaire ?


## 2. Installation et setup

In [1]:

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression


## 3. Chargement des données Olist

In [2]:
print("\n" + "="*50)
print("📁 TÉLÉCHARGEMENT DU DATASET OLIST E-COMMERCE")
print("="*50)

# URL de base du dépôt officiel Olist (miroir Kaggle)
BASE_URL = "https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/"

# Dictionnaire des fichiers principaux
files_urls = {
    "orders": BASE_URL + "olist_orders_dataset.csv",
    "order_items": BASE_URL + "olist_order_items_dataset.csv",
    "products": BASE_URL + "olist_products_dataset.csv",
    "reviews": BASE_URL + "olist_order_reviews_dataset.csv",
    "sellers": BASE_URL + "olist_sellers_dataset.csv",
    "payments": BASE_URL + "olist_order_payments_dataset.csv",
}

# ===============================================
# 🔄 Tentative de chargement des données réelles
# ===============================================

try:
    print("⬇️ Chargement des données réelles Olist...")
    
    df_sellers = pd.read_csv(files_urls["sellers"])
    df_orders = pd.read_csv(files_urls["orders"])
    df_order_items = pd.read_csv(files_urls["order_items"])
    df_products = pd.read_csv(files_urls["products"])
    df_reviews = pd.read_csv(files_urls["reviews"])
    df_payments = pd.read_csv(files_urls["payments"])
    

    print("✅ Données Olist chargées avec succès !")

except Exception as e:
    print("⚠️ Impossible de charger les données réelles.")
    print("👉 Bascule vers des données d'exemple (structure Olist)")
    print(f"ℹ️ Raison : {e}")

# ===============================================
# 📊 Résumé des datasets
# ===============================================

print("\n📊 Résumé des datasets")
print(f"   👥 Sellers      : {len(df_sellers):,}")
print(f"   📦 Orders       : {len(df_orders):,}")
print(f"   🛒 Order items  : {len(df_order_items):,}")
print(f"   📱 Products     : {len(df_products):,}")
print(f"   ⭐ Reviews      : {len(df_reviews):,}")
print(f"   💳 Payments     : {len(df_payments):,}")



📁 TÉLÉCHARGEMENT DU DATASET OLIST E-COMMERCE
⬇️ Chargement des données réelles Olist...
✅ Données Olist chargées avec succès !

📊 Résumé des datasets
   👥 Sellers      : 3,095
   📦 Orders       : 99,441
   🛒 Order items  : 112,650
   📱 Products     : 32,951
   ⭐ Reviews      : 99,224
   💳 Payments     : 103,886


In [5]:
# ===============================================
# 🗃️ 3. CRÉATION DE LA BASE DE DONNÉES DUCKDB
# ===============================================

import duckdb

print("\n" + "="*50)
print("🗃️ CRÉATION DE LA BASE DE DONNÉES SQL")
print("="*50)

# Création d'une connexion DuckDB en mémoire
con = duckdb.connect(database=':memory:')

# Enregistrement des DataFrames pandas comme tables SQL
con.register('sellers', df_sellers)
con.register('orders', df_orders)
con.register('order_items', df_order_items)
con.register('products', df_products)
con.register('reviews', df_reviews)
con.register('payments', df_payments)

print("✅ Base de données créée !")
print("📋 Tables disponibles :")

# Affichage des tables et leurs tailles
tables = ['sellers', 'orders', 'order_items', 'products', 'reviews', 'payments']
for table in tables:
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"   📊 {table}: {count:,} lignes")


🗃️ CRÉATION DE LA BASE DE DONNÉES SQL
✅ Base de données créée !
📋 Tables disponibles :
   📊 sellers: 3,095 lignes
   📊 orders: 99,441 lignes
   📊 order_items: 112,650 lignes
   📊 products: 32,951 lignes
   📊 reviews: 99,224 lignes
   📊 payments: 103,886 lignes


In [7]:
# ===============================================
# 🔍 4. EXPLORATION RAPIDE
# ===============================================

# Requête 1: Aperçu des tables
print("📋 APERÇU DE LA TABLE SELLERS :")
result = con.execute("SELECT * FROM sellers LIMIT 5").df()
print(result)

print("\n📋 APERÇU DE LA TABLE ORDERS :")
result = con.execute("SELECT * FROM orders LIMIT 5").df()
print(result)

# Requête 2: Stats de base
print("\n📊 STATISTIQUES RAPIDES :")
stats_query = """
SELECT
    'order_items' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT seller_id) AS unique_sellers,
    COUNT(DISTINCT order_id) AS unique_orders
FROM order_items

UNION ALL

SELECT
    'orders' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(DISTINCT order_status) AS unique_statuses
FROM orders;

"""

stats_df = con.execute(stats_query).df()
print(stats_df)


📋 APERÇU DE LA TABLE SELLERS :
                          seller_id  seller_zip_code_prefix  \
0  3442f8959a84dea7ee197c632cb2df15                   13023   
1  d1b65fc7debc3361ea86b5f14c68d2e2                   13844   
2  ce3ad9de960102d0677a81f5d0bb7b2d                   20031   
3  c0f3eea2e14555b6faeea3dd58c1b1c3                    4195   
4  51a04a8a6bdcb23deccc82b0b80742cf                   12914   

         seller_city seller_state  
0           campinas           SP  
1         mogi guacu           SP  
2     rio de janeiro           RJ  
3          sao paulo           SP  
4  braganca paulista           SP  

📋 APERÇU DE LA TABLE ORDERS :
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea792

## 4. Les KPI de vendeur

La logique globale est de fournir un tableau de bord des KPI des vendeurs d'e-commerce aux banques pour qu'elles puissent suivre leur performance afin de proposer un crédit personnalisé. 

**Les questions posées**
- Le vendeur génère-t-il assez de revenus ?
- Ses revenus sont-ils stables ?
- Son activité est-elle durable ?
- Est-il fiable opérationnellement ?

Les KPI financiers ont été enrichis par l’intégration des coûts logistiques afin d’approcher la marge réelle du vendeur, élément central dans l’évaluation de sa capacité de remboursement.

In [52]:
print("\n📊 KPI Financiers & Coûts logistiques :")

stats_query = """
CREATE OR REPLACE VIEW seller_financial_kpi_with_payments AS
WITH order_payments AS (
    SELECT
        oi.seller_id,
        oi.order_id,
        SUM(p.payment_value) AS total_paid_per_order,
        COUNT(p.order_id) AS num_payments_per_order,
        SUM(oi.price) - SUM(p.payment_value) AS amount_due_per_order
    FROM order_items oi
    LEFT JOIN payments p
        ON oi.order_id = p.order_id
    GROUP BY oi.seller_id, oi.order_id
)

SELECT 
    oi.seller_id,
    
    -- Revenus
    SUM(oi.price) AS total_revenue,
    AVG(oi.price) AS avg_item_price,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    
    -- Coûts logistiques
    SUM(oi.freight_value) AS total_freight_cost,
    AVG(oi.freight_value) AS avg_freight_per_item,
    
    -- Rentabilité proxy
    SUM(oi.price - oi.freight_value) AS gross_margin_proxy,
    SUM(oi.freight_value) / NULLIF(SUM(oi.price), 0) AS freight_to_revenue_ratio,
    
    -- Paiements / dette clients
    SUM(op.total_paid_per_order) AS total_paid,
    SUM(op.amount_due_per_order) AS cash_gap,
    AVG(op.num_payments_per_order) AS avg_payments_per_order

FROM order_items oi
LEFT JOIN order_payments op
    ON oi.seller_id = op.seller_id AND oi.order_id = op.order_id

GROUP BY oi.seller_id;

"""
con.execute(stats_query)

stats_df = con.execute("SELECT * FROM seller_financial_kpi_with_payments LIMIT 10").df()
print(stats_df)



📊 KPI Financiers & Coûts logistiques :
                          seller_id  total_revenue  avg_item_price  \
0  4d6d651bd7684af3fffabd5f08d12e5a       43587.40      110.347848   
1  4869f7a5dfa277a7dca6462dcf3b52b2      229472.63      198.505735   
2  cc419e0650a3c5ba77189a1882b7556a      104288.42       58.754039   
3  06579cb253ecd5a3a12a9e6eb6bf8f47        4727.20       65.655556   
4  1554a68530182680ad5c8b042c3ab563       29052.53      108.001970   
5  0df3984f9dfb3d49ac6366acbd3bbb85       10021.49      112.601011   
6  9d4db00d65d7760644ac0c14edb5fd86        9979.70       99.797000   
7  b1ac6ea7895bc3dd6f0f6f4abbdd2821        3139.00       76.560976   
8  08633c14ef2db992c11f840f04fad4cd        7644.30       78.807216   
9  a673821011d0cec28146ea42f5ab767f       13863.69       94.956781   

   total_orders  total_freight_cost  avg_freight_per_item  gross_margin_proxy  \
0           369             7898.91             19.997241            35688.49   
1          1132            

## La santé financière des vendeurs 

In [53]:
print("\n📊 CA vendeur et nombre de commandes :")

stats_query = """
CREATE OR REPLACE VIEW seller_order_revenue AS
SELECT
    oi.seller_id,
    SUM(oi.price) AS seller_order_revenue,
    COUNT(oi.order_id) AS total_orders
FROM order_items oi
GROUP BY oi.seller_id
ORDER BY seller_order_revenue DESC;

"""
con.execute(stats_query)

stats_df = con.execute("SELECT * FROM seller_order_revenue LIMIT 10").df()
print(stats_df)




📊 CA vendeur et nombre de commandes :
                          seller_id  seller_order_revenue  total_orders
0  4869f7a5dfa277a7dca6462dcf3b52b2             229472.63          1156
1  53243585a1d6dc2643021fd1853d8905             222776.05           410
2  4a3ca9315b744ce9f8e9374361493884             200472.92          1987
3  fa1c13f2614d7b5c4749cbc52fecda94             194042.03           586
4  7c67e1448b00f6e969d365cea6b010ab             187923.89          1364
5  7e93a43ef30c4f03f38b393420bc753a             176431.87           340
6  da8622b14eb17ae2831f4ac5b9dab84a             160236.57          1551
7  7a67c85e85bb2ce8582c35f2203ad736             141745.53          1171
8  1025f0e2d44d7041d6cf58b6550e0bfa             138968.55          1428
9  955fee9216a65b617aa5c0531780ce60             135171.70          1499


In [ ]:
print("\n📊 CA net :")

stats_query = """
CREATE OR REPLACE VIEW seller_financial_kpi_with_net_income AS
SELECT *,
       -- Résultat net basé sur les paiements réellement reçus
       (total_paid - total_freight_cost) AS net_income_received,

       
       -- Résultat net potentiel basé sur le total des ventes
       (total_revenue - total_freight_cost) AS net_income_potential,
       
       -- Ratio de marge nette
       CASE 
           WHEN total_paid > 0 THEN (total_paid - total_freight_cost) / total_paid
           ELSE NULL
       END AS net_margin_ratio_received,
       
       CASE 
           WHEN total_revenue > 0 THEN (total_revenue - total_freight_cost) / total_revenue
           ELSE NULL
       END AS net_margin_ratio_potential
       
FROM seller_financial_kpi_with_payments;


"""
con.execute(stats_query)

stats_df = con.execute("SELECT * FROM seller_financial_kpi_with_net_income LIMIT 10").df()
print(stats_df)






📊 CA net :
                          seller_id  total_revenue  avg_item_price  \
0  cfb1a033743668a192316f3c6d1d2671       12803.64       69.585000   
1  bfd27a966d91cfaafdb25d076585f0da       19921.00      168.822034   
2  128639473a139ac0f3e5f5ade55873a5       11908.85       21.265804   
3  37be5a7c751166fbc5f8ccba4119e043       55350.55      196.977046   
4  db4350fd57ae30082dec7acbaacc17f9        3331.31       22.817192   
5  7681ef142fd2c19048da7430856b5588       31941.81      414.828701   
6  6d66611d7c44cc30ce351abc49a68421       13593.40       77.235227   
7  9f505651f4a6abe901a56cdc21508025       26361.82       61.306558   
8  d4e4b5192cba4e0e66eb12a9d347239d         441.80       36.816667   
9  37515688008a7a40ac93e3b2e4ab203f        5986.60       24.944167   

   total_orders  total_freight_cost  avg_freight_per_item  gross_margin_proxy  \
0           147             3432.56             18.655217             9371.08   
1           117             2724.05             23.0851

In [55]:
print("\n📊  :")

stats_query = """
CREATE OR REPLACE VIEW seller_financial_kpi_with_net_income_categorized AS
SELECT *,
       CASE
           WHEN net_margin_ratio_potential < 0.10 THEN 'Faible'
           WHEN net_margin_ratio_potential >= 0.10 AND net_margin_ratio_potential < 0.30 THEN 'Moyenne'
           WHEN net_margin_ratio_potential >= 0.30 AND net_margin_ratio_potential < 0.50 THEN 'Bonne'
           ELSE 'Excellente'
       END AS net_margin_category
FROM seller_financial_kpi_with_net_income;
"""
con.execute(stats_query)

stats_df = con.execute("SELECT * FROM seller_financial_kpi_with_net_income_categorized LIMIT 20").df()
print(stats_df)


📊  :
                           seller_id  total_revenue  avg_item_price  \
0   1f50f920176fa81dab994f9023523100      106939.21       55.380223   
1   3d4824f20035949c710eaf111f869d39        1041.04      104.104000   
2   a9b533a26e898b12e8b8d4c07279bf4d         119.60       29.900000   
3   822166ed1e47908f7cfb49946d03c726        3714.83       34.396574   
4   4c1c7281388a33dd06daac44f9fadbd1         188.32       37.664000   
5   813348c996469b40f2e028d5429d3495       11205.83       55.474406   
6   e9779976487b77c6d4ac45f75ec7afe9       43162.95       57.550600   
7   751bdc4d83a466c7206cd42e8f426b03        6648.93       93.646901   
8   5b179e9e8cc7ab6fd113a46ca584da81        5173.50      143.708333   
9   870d0118f7a9d85960f29ad89d5d989a        3157.20       51.757377   
10  7ecd59e5e20407131822c1a68ac59c1f        1466.71       50.576207   
11  0dd184061fb0eaa7ca37932c68ab91c5       18446.92       95.579896   
12  25e6ffe976bd75618accfe16cefcbd0d       11691.61       99.928291   


# Synthèse – Analyse financière et risque de trésorerie des vendeurs (SoLIA)

## 🎯 Objectif
L’objectif de cette analyse est d’évaluer la **rentabilité**, la **liquidité** et le **risque de trésorerie à court terme** des vendeurs e-commerce, afin d’aider une banque ou une FinTech à :
- décider de l’octroi d’un prêt,
- identifier les vendeurs à risque,
- estimer le besoin de financement lié aux paiements fractionnés des clients.

L’approche repose sur des **indicateurs financiers explicables**, construits à partir des ventes, des coûts logistiques et des paiements séquentiels (`payment_sequential`).

---

## 🧱 Données utilisées
Les indicateurs sont calculés à partir des tables :
- `sellers` : base des vendeurs
- `order_items` : ventes et coûts logistiques
- `payments` : paiements clients (possiblement en plusieurs fois)

Les paiements sont supposés **mensuels**, et la variable `payment_sequential` est utilisée comme **proxy temporel** (en mois).

---

## Indicateurs financiers clés

### 1️⃣ Chiffre d’affaires et coûts
- **`total_revenue`** : chiffre d’affaires total du vendeur  
- **`total_freight_cost`** : coûts logistiques totaux  

Ces indicateurs décrivent la **taille de l’activité** et les **charges incompressibles**.

---

### 2️⃣ Résultats nets
- **Résultat net potentiel**  

    net_income_potential = total_revenue - total_freight_cost

➡️ Rentabilité théorique si tous les clients paient.

- **Résultat net encaissé**  
    
    net_income_received = total_paid - total_freight_cost

➡️ Rentabilité réelle en trésorerie.

---

### 3️⃣ Cash gap (dette client court terme)

    cash_gap = total_revenue - total_paid

➡️ Montant du chiffre d’affaires **non encore encaissé**, correspondant à une **dette client**.  
C’est l’indicateur central du **besoin de financement court terme**.

---

### 4️⃣ Temps moyen d’encaissement
- **`avg_payments_per_order`** = moyenne de `payment_sequential` 

➡️ Mesure la durée pendant laquelle la trésorerie est immobilisée chez les clients.

---

### 5️⃣ Marges nettes
- **Marge nette potentielle**

    net_margin_ratio_potential = net_income_potential/total_revenue

- **Marge nette encaissée**

    net_margin_ratio_received = net_income_received/total_paid

➡️ Capacité du vendeur à absorber des retards de paiement.

---

## 🧩 Catégorisation de la rentabilité

Les vendeurs sont classés selon leur **marge nette potentielle** :

| Catégorie | Seuil |
|---------|------|
| Faible | < 10 % |
| Moyenne | 10 – 30 % |
| Bonne | 30 – 50 % |
| Excellente | > 50 % |

Cette catégorisation permet une **lecture rapide** de la solidité économique.

---

## Lecture bancaire globale

- **Rentabilité** → marge nette potentielle  
- **Liquidité** → cash gap + temps d’encaissement  
- **Risque court terme** → écart entre net income potentiel et encaissé  

Un vendeur peut être **rentable mais risqué** s’il finance trop longtemps ses clients.

---

## Conclusion
Cette approche permet :
- une évaluation **fine et explicable** du risque vendeur,
- une distinction claire entre **performance économique** et **problème de trésorerie**,
- une base solide pour un **score de solvabilité** ou une **décision de crédit automatique**.

Elle est particulièrement adaptée aux vendeurs e-commerce avec **revenus fluctuants et paiements fractionnés**, cible principale du projet SoLIA.
